In [1]:
import json
import numpy as np
import pandas as pd

DATA_PATH = "cleaned_dataset.csv"
PARAMS_PATH = "bernoulli_params.npz"
VOCAB_PATH = "bernoulli_vocab.json"
TEXT_COLS = ["feel_text", "food", "soundtrack"]


def fill_missing_text(df, columns):
    for col in columns:
        df[col] = df[col].fillna("")


def build_binary_matrix(texts, vocab):
    X = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    for i, text in enumerate(texts):
        for word in str(text).split():
            if word in vocab:
                X[i, vocab[word]] = 1.0
    return X


def load_artifacts(params_path=PARAMS_PATH, vocab_path=VOCAB_PATH):
    params = np.load(params_path, allow_pickle=True)
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab_data = json.load(f)

    return {
        "class_order": params["class_order"],
        "class_log_prior": np.asarray(params["class_log_prior"], dtype=np.float64),
        "feature_log_prob": np.asarray(params["feature_log_prob"], dtype=np.float64),
        "neg_log_prob": np.asarray(params["neg_log_prob"], dtype=np.float64),
        "vocab_data": vocab_data,
    }


def build_feature_matrix(df, vocab_data):
    text_blocks = []
    for col in TEXT_COLS:
        vocab = {word: int(idx) for word, idx in vocab_data[col]["vocab"].items()}
        text_blocks.append(build_binary_matrix(df[col], vocab))
    return np.hstack(text_blocks)


def predict_bernoulli_nb(X, model):
    log_scores = np.tile(model["class_log_prior"], (X.shape[0], 1))
    log_scores += X @ model["feature_log_prob"].T
    log_scores += (1.0 - X) @ model["neg_log_prob"].T
    pred_positions = np.argmax(log_scores, axis=1)
    return model["class_order"][pred_positions]


def accuracy(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))


df = pd.read_csv(DATA_PATH)
fill_missing_text(df, TEXT_COLS)

PAINTING_NAMES = sorted(df["painting"].dropna().unique())
df["target"] = df["painting"].map({name: i for i, name in enumerate(PAINTING_NAMES)})

print(f"Loaded {len(df)} rows")
print("Classes:", PAINTING_NAMES)
print(df["target"].value_counts())

np.random.seed(42)
train_idx, val_idx = [], []

for cls in df["target"].unique():
    idx = df[df["target"] == cls].index.tolist()
    np.random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_idx.extend(idx[:cut])
    val_idx.extend(idx[cut:])

X_train = df.loc[train_idx].reset_index(drop=True)
X_val = df.loc[val_idx].reset_index(drop=True)
y_train = X_train["target"].values
y_val = X_val["target"].values

print(f"Train: {len(X_train)}  |  Val: {len(X_val)}")

model = load_artifacts()
X_train_combined = build_feature_matrix(X_train, model["vocab_data"])
X_val_combined = build_feature_matrix(X_val, model["vocab_data"])

print(f"Feature matrix shape: {X_train_combined.shape}")

train_preds = predict_bernoulli_nb(X_train_combined, model)
val_preds = predict_bernoulli_nb(X_val_combined, model)

print(f"Train acc: {accuracy(y_train, train_preds):.4f}")
print(f"Val   acc: {accuracy(y_val, val_preds):.4f}")

print("Per-class breakdown:")
for idx, name in enumerate(PAINTING_NAMES):
    mask = y_val == idx
    if mask.sum() > 0:
        acc_c = accuracy(y_val[mask], val_preds[mask])
        print(f"  {name:35s}: {acc_c:.4f}  (n={mask.sum()})")


Loaded 1610 rows
Classes: ['The Persistence of Memory', 'The Starry Night', 'The Water Lily Pond']
target
0    541
1    535
2    534
Name: count, dtype: int64
Train: 1287  |  Val: 323
Feature matrix shape: (1287, 1305)
Train acc: 0.8531
Val   acc: 0.7709
Per-class breakdown:
  The Persistence of Memory          : 0.7248  (n=109)
  The Starry Night                   : 0.6636  (n=107)
  The Water Lily Pond                : 0.9252  (n=107)
